# Chapter 06：Fused Softmax

**目标**：实现 row-wise fused softmax。核心概念是数值稳定的 `x - max(x)`，以及把 load、两次 reduction、除法和 store 放入一个 kernel。

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name.startswith("chapter_"):
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
import triton
import triton.language as tl

from common.benchmark import bench
from common.check import assert_close
from common.utils import get_device, set_seed

device = get_device()
set_seed(0)

## 1. 公式与 PyTorch reference

对每行 `x`，softmax 为 `exp(x_i) / sum(exp(x_j))`。直接指数运算可能溢出；减去该行最大值不改变最终比例，却显著提升稳定性。

In [ ]:
M, N = 1024, 513
x = torch.randn(M, N, device=device)
torch_output = torch.softmax(x, dim=1)
torch_output[0, :5]

## 2. Triton fused softmax kernel

一个 program 处理完整一行。无效 lane load 为负无穷，减 max 后指数变为 0，因此不会影响分母。store 仍用 mask。

In [ ]:
MAX_FUSED_SIZE = 65_536

@triton.jit
def softmax_kernel(x_ptr, output_ptr, n_cols, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols
    values = tl.load(x_ptr + row * n_cols + cols, mask=mask, other=-float("inf"))
    stable_values = values - tl.max(values, axis=0)
    numerator = tl.exp(stable_values)
    denominator = tl.sum(numerator, axis=0)
    output = numerator / denominator
    tl.store(output_ptr + row * n_cols + cols, output, mask=mask)

## 3. Python wrapper 与宽度限制

wrapper 选择 `next_power_of_2(N)`。教学 kernel 把 block 上限设为 65,536；更宽的行应改用分块或多阶段算法。

In [ ]:
def softmax(x):
    if x.ndim != 2 or not x.is_cuda or not x.is_contiguous():
        raise ValueError("softmax expects a contiguous 2D CUDA tensor")
    M, N = x.shape
    if M == 0 or N == 0:
        raise ValueError("M and N must be positive")
    block_size = triton.next_power_of_2(N)
    if block_size > MAX_FUSED_SIZE:
        raise ValueError(
            f"N={N} is too large; next power of two must be <= {MAX_FUSED_SIZE}"
        )
    output = torch.empty_like(x)
    num_warps = 8 if block_size >= 2048 else 4
    softmax_kernel[(M,)](
        x, output, N, BLOCK_SIZE=block_size, num_warps=num_warps
    )
    return output

## 4. Correctness check

除了逐元素接近 PyTorch，也检查每行概率和接近 1。

In [ ]:
triton_output = softmax(x)
assert_close("fused softmax", triton_output, torch_output, rtol=1e-3, atol=1e-5)
assert_close("row probability sums", triton_output.sum(dim=1), torch.ones(M, device=device))

## 5. Benchmark

PyTorch 可能也使用高度优化的 fused 实现，因此结果取决于 GPU、版本和 shape。这里的目标是学习，而不是宣称普遍更快。

In [ ]:
for shape in ((256, 127), (1024, 512), (512, 1024)):
    sample = torch.randn(*shape, device=device)
    print(
        f"shape={shape}: torch={bench(lambda: torch.softmax(sample, dim=1)):.3f} ms, "
        f"triton={bench(lambda: softmax(sample)):.3f} ms"
    )

## 为什么 fusion 减少内存访问

概念上的分步实现会产生 row max、shifted values、exp values 和 row sum 等中间结果。这个 kernel 将它们保留在 program 的寄存器/片上处理中，只从全局内存读取输入并写出最终输出。

**练习**：将输入整体加上 100，再比较稳定版本与 PyTorch；结果仍应正确且不出现 `inf`。

In [ ]:
shifted = x + 100.0
assert_close(
    "large shifted softmax",
    softmax(shifted),
    torch.softmax(shifted, dim=1),
    rtol=1e-3,
    atol=1e-5,
)